# Baseline Models
This notebook establishes a naive tracking baseline and a Ridge regression model for the log returns.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score   
from pathlib import Path

# Handle paths based on execution directory (whether run from root or inside notebooks folder)
if Path.cwd().name == 'notebooks':
    data_dir = Path('../data/processed')
else:
    data_dir = Path('data/processed')

train_path = data_dir / 'train.csv'
test_path = data_dir / 'test.csv'
print(f"Reading data from: {data_dir}")

Reading data from: ../data/processed


## 1. Load Data

In [2]:
print("Loading data...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Sort by Region and Date chronologically to strictly respect time
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])
train_df = train_df.sort_values(['RegionID', 'date']).reset_index(drop=True)
test_df = test_df.sort_values(['RegionID', 'date']).reset_index(drop=True)

features = ['lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'city_enc']
target = 'log_return'

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

Loading data...


## 2. Naive Baseline
Predicts $t+1$ using strictly `lag_1`.

In [3]:
naive_preds_train = X_train['lag_1']
naive_preds_test = X_test['lag_1']

naive_r2_log = r2_score(y_test, naive_preds_test)
naive_mae_log = mean_absolute_error(y_test, naive_preds_test)
naive_rmse_log = np.sqrt(mean_squared_error(y_test, naive_preds_test))
print(f"[Naive] log_return MAE: {naive_mae_log:.6f} | RMSE: {naive_rmse_log:.6f} | R2: {naive_r2_log:.6f}")

[Naive] log_return MAE: 1.103105 | RMSE: 1.488839 | R2: -36053.929685


ridge = Ridge(random_state=42)
param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

# TimeSeriesSplit prevents validation leakage into the future
tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=ridge, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_ridge = grid_search.best_estimator_
print(f"Best Ridge Alpha: {best_ridge.alpha}")

ridge_preds_test = best_ridge.predict(X_test)
ridge_r2_log = r2_score(y_test, ridge_preds_test)
ridge_mae_log = mean_absolute_error(y_test, ridge_preds_test)
ridge_rmse_log = np.sqrt(mean_squared_error(y_test, ridge_preds_test))
print(f"[Ridge] log_return MAE: {ridge_mae_log:.6f} | RMSE: {ridge_rmse_log:.6f} | R2: {ridge_r2_log:.6f}")


In [4]:
ridge = Ridge(random_state=42)
param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

# TimeSeriesSplit prevents validation leakage into the future
tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=ridge, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_ridge = grid_search.best_estimator_
print(f"Best Ridge Alpha: {best_ridge.alpha}")

ridge_preds_test = best_ridge.predict(X_test)
ridge_r2_log = r2_score(y_test, naive_preds_test)
ridge_mae_log = mean_absolute_error(y_test, ridge_preds_test)
ridge_rmse_log = np.sqrt(mean_squared_error(y_test, ridge_preds_test))
print(f"[Ridge] log_return MAE: {ridge_mae_log:.6f} | RMSE: {ridge_rmse_log:.6f} | R2: {ridge_r2_log}")

Best Ridge Alpha: 0.001
[Ridge] log_return MAE: 0.002009 | RMSE: 0.002756 | R2: -36053.92968464434


## 4. Inverse Transform & Price Evaluation

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Re-derive the previous price using log relations: Price_{t-1} = Price_t / exp(actual_log_return_t)
test_df['price_prev'] = test_df['price'] / np.exp(test_df['log_return'])

# Formulate current price predictions mapping log returns back to raw estimates
test_df['naive_price_pred'] = test_df['price_prev'] * np.exp(naive_preds_test)
test_df['ridge_price_pred'] = test_df['price_prev'] * np.exp(ridge_preds_test)

naive_price_mae = mean_absolute_error(test_df['price'], test_df['naive_price_pred'])
ridge_price_mae = mean_absolute_error(test_df['price'], test_df['ridge_price_pred'])

naive_price_rmse = np.sqrt(mean_squared_error(test_df['price'], test_df['naive_price_pred']))
ridge_price_rmse = np.sqrt(mean_squared_error(test_df['price'], test_df['ridge_price_pred']))

naive_price_r2 = r2_score(test_df['price'], test_df['naive_price_pred'])
ridge_price_r2 = r2_score(test_df['price'], test_df['ridge_price_pred'])

improvement_mae = naive_price_mae - ridge_price_mae
improvement_rmse = naive_price_rmse - ridge_price_rmse
pct_improvement = (improvement_mae / naive_price_mae) * 100 if naive_price_mae > 0 else 0

result_str = f"[Naive Baseline] log_return - MAE: {naive_mae_log:.6f} | RMSE: {naive_rmse_log:.6f} | R2: {naive_r2_log:.6f}\n"
result_str += f"[Ridge Baseline] log_return - MAE: {ridge_mae_log:.6f} | RMSE: {ridge_rmse_log:.6f} | R2: {ridge_r2_log:.6f}\n"
result_str += "\n"
result_str += f"[Naive Baseline] Price Space - MAE: ${naive_price_mae:,.2f} | RMSE: ${naive_price_rmse:,.2f} | R2: {naive_price_r2:.6f}\n"
result_str += f"[Ridge Baseline] Price Space - MAE: ${ridge_price_mae:,.2f} | RMSE: ${ridge_price_rmse:,.2f} | R2: {ridge_price_r2:.6f}\n"
result_str += f"\n--> Ridge model improves Price MAE by ${improvement_mae:,.2f} ({pct_improvement:.2f}%) over Naive Baseline.\n"
print(result_str)

# Save to assets
if Path.cwd().name == 'notebooks':
    assets_dir = Path('../assets')
else:
    assets_dir = Path('assets')
assets_dir.mkdir(parents=True, exist_ok=True)
with open(assets_dir / 'baseline_metrics.txt', 'w') as f:
    f.write(result_str)


[Naive Baseline] log_return - MAE: 1.103105 | RMSE: 1.488839 | R2: -36053.929685
[Ridge Baseline] log_return - MAE: 0.002009 | RMSE: 0.002756 | R2: -36053.929685

[Naive Baseline] Price Space - MAE: $1,554,653.37 | RMSE: $16,024,956.79 | R2: -9509.026457
[Ridge Baseline] Price Space - MAE: $495.26 | RMSE: $804.76 | R2: 0.999976

--> Ridge model improves Price MAE by $1,554,158.11 (99.97%) over Naive Baseline.

